# <a id="dataset-dataloader"></a>
## 1. Création du Dataset PyTorch et de la Fonction Collate (Padding)

Les phrases n'ont pas toutes la même longueur, mais PyTorch a besoin de tenseurs de forme fixe pour faire du calcul par batch. La solution standard : on regroupe des phrases dans un Dataset/DataLoader, et une fonction collate_fn ajoute du padding (avec 0, l'index de <PAD>) pour aligner toutes les phrases d'un même batch sur la longueur de la plus longue.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class POSDataset(Dataset):
    """
    Dataset PyTorch personnalisé pour stocker les séquences de mots et d'étiquettes.
    """
    def __init__(self, data, word_to_ix, tag_to_ix):
        self.samples = []
        for words, tags in data:
            word_idxs = torch.tensor([word_to_ix.get(w, word_to_ix["<UNK>"]) for w in words], dtype=torch.long)
            tag_idxs = torch.tensor([tag_to_ix[t] for t in tags], dtype=torch.long)
            self.samples.append((word_idxs, tag_idxs))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def pad_collate_fn(batch):
    """
    Fonction de regroupement (collate) pour aligner les longueurs des séquences avec <PAD>=0.
    """
    sequences, labels = zip(*batch)
    sequences_padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=0)
    return sequences_padded, labels_padded

# <a id="training-loop"></a>
## 2. Boucle d'Entraînement avec Calcul de Perte (Masking <PAD>)

In [2]:
def train_model(model, dataloader, optimizer, criterion, epochs=10, device="cpu"):
    """
    Effectue l'entraînement du modèle sur plusieurs époques.
    """
    model.to(device)
    model.train()
    
    for epoch in range(epochs):
        total_loss = 0.0
        for sentences, targets in dataloader:
            sentences, targets = sentences.to(device), targets.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(sentences) # (batch_size, seq_len, tagset_size)
            
            # Repasser sous forme 2D pour le calcul de la CrossEntropyLoss
            # (batch_size * seq_len, tagset_size) vs (batch_size * seq_len)
            loss = criterion(outputs.view(-1, outputs.shape[-1]), targets.view(-1))
            
            # Backward pass & Optimisation
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        print(f"Époque [{epoch+1}/{epochs}] - Perte (Loss): {total_loss / len(dataloader):.4f}")

# <a id="pipeline-test"></a>
## 3. Test de la Pipeline d'Entraînement Complète

In [5]:
# Données fictives et vocabulaires
training_data = [
    ("Amadou étudie le bambara".split(), ["NOM", "VERBE", "DET", "NOM"]),
    ("le modèle comprend les phrases".split(), ["DET", "NOM", "VERBE", "DET", "NOM"]),
    ("un autre exemple".split(), ["DET", "ADJ", "NOM"])
]

word_to_ix = {"<PAD>": 0, "<UNK>": 1}
for sentence, _ in training_data:
    for word in sentence:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)

tag_to_ix = {"<PAD>": 0, "NOM": 1, "VERBE": 2, "DET": 3, "ADJ": 4}

# Initialisation Dataset et DataLoader
dataset = POSDataset(training_data, word_to_ix, tag_to_ix)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=pad_collate_fn)

# Import du modèle (réutilisation de la classe AdvancedBiLSTMTagger)
class AdvancedBiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim, hidden_dim):
        super(AdvancedBiLSTMTagger, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=2, bidirectional=True, batch_first=True)
        self.linear = nn.Linear(hidden_dim * 2, tagset_size)
    
    def forward(self, x):
        x = self.embedding(x)
        lstm_out, _ = self.lstm(x)
        return self.linear(lstm_out)

model = AdvancedBiLSTMTagger(
    vocab_size=len(word_to_ix),
    tagset_size=len(tag_to_ix),
    embedding_dim=32,
    hidden_dim=64
)

# Configuration de l'optimiseur et de la perte en ignorant l'index de padding (0)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# Lancement de l'entraînement de test
print("--- Démarrage du Test d'Entraînement ---")
train_model(model, dataloader, optimizer, criterion, epochs=10 , device="cpu")

--- Démarrage du Test d'Entraînement ---
Époque [1/10] - Perte (Loss): 1.6061
Époque [2/10] - Perte (Loss): 1.2893
Époque [3/10] - Perte (Loss): 1.1166
Époque [4/10] - Perte (Loss): 0.8403
Époque [5/10] - Perte (Loss): 0.5637
Époque [6/10] - Perte (Loss): 0.2795
Époque [7/10] - Perte (Loss): 0.1162
Époque [8/10] - Perte (Loss): 0.0467
Époque [9/10] - Perte (Loss): 0.0144
Époque [10/10] - Perte (Loss): 0.0047
